## all_players_pergame_cleaning
- prepping Lebron, Jordan, and Kobe's per_game csv's for further SQL querying

Remember:
- Pandas is Data Cleaning
- Matplotlib is Data visuals
- Numpy is Math Operations/Calculator

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import psycopg2
from pandasql import sqldf
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
pd.options.plotting.backend = 'plotly'
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
import voila
import warnings
warnings.filterwarnings('ignore')

In [11]:
#resuable function recipe for loading csv
def clean_per_game(file_path, player_id):
    df = pd.read_csv(file_path)

    # 1. keep only real seasons (rows where Age is all digits)
    df = df[df['Age'].str.isdigit().fillna(False)]

    # 2. convert the number columns
    text_cols = ['Season', 'Team', 'Lg', 'Pos', 'Awards']
    num_cols = [col for col in df.columns if col not in text_cols]
    df[num_cols] = df[num_cols].apply(pd.to_numeric)

    # 3. rename the columns so they are SQL friendly
    df.columns = (
        df.columns
        .str.lower()                   #FG% to fg%
        .str.replace('%', '_pct')      #fg% to fg_pct
        .str.replace('3p', 'three_p')  #3p to three_p
        .str.replace('2p', 'two_p')    #2p to two_p
    )

    # 4. add the player_id column (1 = LeBron, 2 = Jordan, 3 = Kobe)
    df['player_id'] = player_id

    return df

In [3]:
lebron = clean_per_game('../data/raw/lebron_per_game.csv', 1)
jordan = clean_per_game('../data/raw/jordan_per_game.csv', 2)
kobe   = clean_per_game('../data/raw/kobe_per_game.csv', 3)

In [4]:
#check each player: rows should be 23, 15 and 20, with 32 columns each
lebron.shape, jordan.shape, kobe.shape

((23, 32), (15, 32), (20, 32))

In [5]:
#pd.concat stacks the three data frames together. to make one big dataframe. 
#concat is bringing and joining together
#similar to union all in SQL
all_players = pd.concat([lebron, jordan, kobe], ignore_index=True)
all_players

,season,age,team,lg,pos,g,gs,mp,fg,fga,...,drb,trb,ast,stl,blk,tov,pf,pts,awards,player_id
0,2003-04,19,CLE,NBA,SG,79,79,39.5,7.9,18.9,...,4.2,5.5,5.9,1.6,0.7,3.5,1.9,20.9,MVP-9ROY-1,1
1,2004-05,20,CLE,NBA,SF,80,80,42.4,9.9,21.1,...,6.0,7.4,7.2,2.2,0.7,3.3,1.8,27.2,MVP-6ASNBA2,1
2,2005-06,21,CLE,NBA,SF,79,79,42.5,11.1,23.1,...,6.1,7.0,6.6,1.6,0.8,3.3,2.3,31.4,MVP-2ASNBA1,1
3,2006-07,22,CLE,NBA,SF,78,78,40.9,9.9,20.8,...,5.7,6.7,6.0,1.6,0.7,3.2,2.2,27.3,MVP-5ASNBA2,1
4,2007-08,23,CLE,NBA,SF,75,74,40.4,10.6,21.9,...,6.1,7.9,7.2,1.8,1.1,3.4,2.2,30.0,MVP-4ASNBA1,1
5,2008-09,24,CLE,NBA,SF,81,81,37.7,9.7,19.9,...,6.3,7.6,7.2,1.7,1.1,3.0,1.7,28.4,MVP-1DPOY-2ASNBA1DEF1,1
6,2009-10,25,CLE,NBA,SF,76,76,39.0,10.1,20.1,...,6.4,7.3,8.6,1.6,1.0,3.4,1.6,29.7,MVP-1DPOY-4ASNBA1DEF1,1
7,2010-11,26,MIA,NBA,SF,79,79,38.8,9.6,18.8,...,6.5,7.5,7.0,1.6,0.6,3.6,2.1,26.7,MVP-3DPOY-9ASNBA1DEF1,1
8,2011-12,27,MIA,NBA,SF,62,62,37.5,10.0,18.9,...,6.4,7.9,6.2,1.9,0.8,3.4,1.5,27.1,MVP-1DPOY-4ASNBA1DEF1,1
9,2012-13,28,MIA,NBA,PF,76,76,37.9,10.1,17.8,...,6.8,8.0,7.3,1.7,0.9,3.0,1.4,26.8,MVP-1DPOY-2ASNBA1DEF1,1


### Check the combined table

In [6]:
#should be (58, 32)
all_players.shape

(58, 32)

In [7]:
#how many seasons each player has
all_players['player_id'].value_counts()
#lebron 23 years, Jordan 15 years, Kobe 20 years

player_id
1    23
3    20
2    15
Name: count, dtype: int64

In [8]:
#every stat column should be a number
all_players.info()

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 32 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   season       58 non-null     str    
 1   age          58 non-null     int64  
 2   team         58 non-null     str    
 3   lg           58 non-null     str    
 4   pos          58 non-null     str    
 5   g            58 non-null     int64  
 6   gs           58 non-null     int64  
 7   mp           58 non-null     float64
 8   fg           58 non-null     float64
 9   fga          58 non-null     float64
 10  fg_pct       58 non-null     float64
 11  three_p      58 non-null     float64
 12  three_pa     58 non-null     float64
 13  three_p_pct  58 non-null     float64
 14  two_p        58 non-null     float64
 15  two_pa       58 non-null     float64
 16  two_p_pct    58 non-null     float64
 17  efg_pct      58 non-null     float64
 18  ft           58 non-null     float64
 19  fta          58 non-n